# RST: evaluate the judge and calibrate the AI difficulty levels

Run this **after** `RST_train_association_model.ipynb`. It needs `rst_artifacts.zip`
(the notebook's download), and it runs fine on a CPU runtime.

1. Load the artifacts with the **same numpy code the game server uses**
2. Try the judge interactively
3. Simulate thousands of bot-vs-bot games to measure, for each difficulty level,
   the mistake rate, link strength, and head-to-head win rate

## 1. Serving code (copied verbatim from the repo)

In [ ]:
%%writefile rst_nlp.py
"""Serve the trained association model with numpy only (no torch at runtime).

Artifact directory layout (produced by notebooks/RST_train_association_model.ipynb):

  meta.json    {"version", "judge": "mlp" | "cosine", "platt": [a, b],
                "thresholds": {"lenient","normal","strict"}, "metrics": {...}}
  vocab.json   {"words": [...], "lemma": {word: lemma}, "playable": [0/1 per word],
                "seeds": [...]}
  emb.npy      float16 [N, d]  fine-tuned word embeddings (L2-normalised)
  z.npy        float16 [N, Z]  judge space          (optional; defaults to emb)
  q.npy, k.npy float16 [N, Z]  directional player space: score(a->b) = q[a].k[b]
  judge.npz    MLP over [z_a*z_b, |z_a-z_b|, cos(e_a,e_b)]: W1, b1, W2, b2, W3, b3,
               plus optional w_cos: logit = mlp(x) + w_cos * cos(e_a, e_b)
  known_pairs.npy  (optional) int32 [M, 2] attested human/ConceptNet associations;
               these are accepted outright, the neural judge decides everything else.
  hub.npy      (optional) float16 [N] hubness from scripts/compute_hubness.py; a novel
               pair scores p - meta.hub.beta * hub[word] so generic words can't be spammed.

The judge is symmetric. Its probability is sigmoid(platt_a * logit + platt_b);
the cosine fallback (baseline artifacts) is sigmoid(a * cos + b) with a, b = platt.
"""
from __future__ import annotations

import json
import random
from functools import lru_cache
from pathlib import Path

import numpy as np

FORBIDDEN = ("r", "s", "t")
KNOWN_PAIR_PROB = 0.95  # attested associations pass even the strict threshold


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


class AssociationModel:
    def __init__(self, artifact_dir: str | Path):
        d = Path(artifact_dir)
        self.meta = json.loads((d / "meta.json").read_text())
        vocab = json.loads((d / "vocab.json").read_text())
        self.words: list[str] = vocab["words"]
        self.index = {w: i for i, w in enumerate(self.words)}
        self.lemma: dict[str, str] = vocab.get("lemma", {})
        self.playable = np.asarray(vocab.get("playable", [1] * len(self.words)), dtype=bool)
        self.seeds = [w for w in vocab.get("seeds", []) if w in self.index and w[0] not in FORBIDDEN
                      and self.lemma.get(w, w) == w]  # "hat", not "hats"
        self.E = np.load(d / "emb.npy").astype(np.float32)
        load = lambda name: np.load(d / name).astype(np.float32) if (d / name).exists() else self.E
        self.Z, self.Q, self.K = load("z.npy"), load("q.npy"), load("k.npy")
        self.forbidden_mask = np.array([w[0] in FORBIDDEN for w in self.words])
        self.judge = None
        if self.meta.get("judge") == "mlp" and (d / "judge.npz").exists():
            self.judge = dict(np.load(d / "judge.npz"))
        # Retrieval only ever returns playable words, so search just those rows.
        self.play_idx = np.flatnonzero(self.playable)
        self.K_play = np.ascontiguousarray(self.K[self.play_idx])
        self.play_forbidden = self.forbidden_mask[self.play_idx]
        # Known pairs as sorted int64 keys a*N+b (both directions) for vectorised lookup.
        n = len(self.words)
        self.known = np.zeros(0, dtype=np.int64)
        if (d / "known_pairs.npy").exists():
            kp = np.load(d / "known_pairs.npy").astype(np.int64)
            self.known = np.unique(np.concatenate([kp[:, 0] * n + kp[:, 1], kp[:, 1] * n + kp[:, 0]]))
        self.hub = np.load(d / "hub.npy").astype(np.float32) if (d / "hub.npy").exists() else None
        self.hub_beta = float(self.meta.get("hub", {}).get("beta", 0.0))
        self.platt = self.meta.get("platt") or self.meta.get("cosine_calib", [8.0, -3.0])
        self._thr = self.meta["thresholds"]

    # ------------------------------------------------------------ lexicon
    def in_vocab(self, w: str) -> bool:
        return w in self.index

    def normalize(self, w: str) -> str:
        return self.lemma.get(w, w)

    def threshold(self, strictness: str) -> float:
        return float(self._thr.get(strictness, self._thr["normal"]))

    def random_seed_word(self) -> str:
        if self.seeds:
            return random.choice(self.seeds)
        pool = np.flatnonzero(self.playable[:3000] & ~self.forbidden_mask[:3000])
        return self.words[int(random.choice(pool))]

    # ------------------------------------------------------------- scoring
    def _score(self, i: int, js: np.ndarray) -> np.ndarray:
        """Game score for prev word i -> each candidate j: attested pairs pass outright,
        novel pairs pay a hubness penalty for the played word."""
        js = np.asarray(js)
        p = self._pair_prob(np.full(len(js), i), js)
        if self.hub is not None:
            p = np.clip(p - self.hub_beta * self.hub[js], 0.0, 1.0)
        return np.where(self._is_known(i, js), np.maximum(p, KNOWN_PAIR_PROB), p)

    def _is_known(self, i: int, js: np.ndarray) -> np.ndarray:
        if not len(self.known):
            return np.zeros(len(js), dtype=bool)
        keys = i * len(self.words) + np.asarray(js, dtype=np.int64)
        pos = np.minimum(np.searchsorted(self.known, keys), len(self.known) - 1)
        return self.known[pos] == keys

    def _pair_prob(self, ia: np.ndarray, ib: np.ndarray) -> np.ndarray:
        """Index arrays [n] -> [n] judge probabilities."""
        a, b = self.platt
        cos = np.sum(self.E[ia] * self.E[ib], axis=1, keepdims=True)
        if self.judge is None:
            return _sigmoid(a * cos[:, 0] + b)
        J, za, zb = self.judge, self.Z[ia], self.Z[ib]
        x = np.concatenate([za * zb, np.abs(za - zb), cos], axis=1)
        h = np.maximum(0, x @ J["W1"] + J["b1"])
        h = np.maximum(0, h @ J["W2"] + J["b2"])
        logit = (h @ J["W3"] + J["b3"])[:, 0]
        if "w_cos" in J:
            logit = logit + float(J["w_cos"].reshape(-1)[0]) * cos[:, 0]
        return _sigmoid(a * logit + b)

    @lru_cache(maxsize=50_000)
    def relatedness(self, a: str, b: str) -> float:
        if a not in self.index or b not in self.index:
            return 0.0
        if self.normalize(a) == self.normalize(b):
            return 0.0
        return float(self._score(self.index[a], np.array([self.index[b]]))[0])

    def candidates(self, prev: str, exclude: set[str] = frozenset(), k: int = 50,
                   allow_forbidden: bool = False, retrieve: int = 200) -> list[tuple[str, float]]:
        """Top-k playable next words for `prev`, in the order a person would think of
        them (player head), each with its judge probability."""
        if prev not in self.index:
            return []
        i = self.index[prev]
        sims = self.K_play @ self.Q[i]
        if not allow_forbidden:
            sims[self.play_forbidden] = -np.inf
        retrieve = min(retrieve, len(sims) - 1)
        top = np.argpartition(-sims, retrieve)[:retrieve]
        top = top[np.isfinite(sims[top])]
        top = self.play_idx[top[np.argsort(-sims[top])]]
        top = top[top != i]
        probs = self._score(i, top)
        prev_norm = self.normalize(prev)
        out, seen = [], set()
        for j, pr in zip(top, probs):
            w = self.words[int(j)]
            n = self.normalize(w)
            if n in exclude or n in seen or n == prev_norm or prev_norm in w or w in prev:
                continue
            seen.add(n)
            out.append((w, float(pr)))
            if len(out) >= k:
                break
        return out

    def forbidden_pressure_many(self, words: list[str], used: set[str], k: int = 20) -> np.ndarray:
        """For each word, the share of its k most natural follow-ups (player head) that the
        next player cannot use: banned initial letter or already played. One matmul."""
        idx = np.array([self.index[w] for w in words])
        sims = self.K_play @ self.Q[idx].T                      # [P, n]
        top = np.argpartition(-sims, k + 4, axis=0)[: k + 4]    # a few spare for self/lemma hits
        out = np.zeros(len(words))
        for c, i in enumerate(idx):
            col = top[:, c][np.argsort(-sims[top[:, c], c])]
            norm_i = self.normalize(self.words[i])
            picks = [self.words[j] for j in self.play_idx[col]
                     if self.normalize(self.words[j]) != norm_i][:k]
            out[c] = sum(w[0] in FORBIDDEN or self.normalize(w) in used for w in picks) / max(1, len(picks))
        return out

    def forbidden_pressure(self, w: str, used: set[str], k: int = 20) -> float:
        return float(self.forbidden_pressure_many([w], used, k)[0])

In [ ]:
%%writefile rst_bots.py
"""AI players.

Each difficulty controls three things:
  * how deep in the model's ranked candidate list the bot picks from,
  * how quickly it answers,
  * how often it makes a human-like blunder (an R/S/T word, a repeat, freezing).

Grandmaster also plays strategically: among strong candidates it prefers words
whose own best associations mostly start with R/S/T, leaving the next player
few good moves.
"""
from __future__ import annotations

import asyncio
import random
import time
from dataclasses import dataclass


@dataclass(frozen=True)
class Profile:
    rank_lo: int        # pick among candidates ranked [rank_lo, rank_hi)
    rank_hi: int
    think: tuple[float, float]  # seconds
    blunder: float      # chance of a mistake on any turn
    strategic: bool = False
    vocab_cap: int | None = None  # only words this common (frequency rank); None = any


PROFILES = {
    "rookie":      Profile(10, 120, (4.0, 9.0), 0.22, vocab_cap=6000),
    "casual":      Profile(3, 50,  (3.0, 6.5), 0.10, vocab_cap=10000),
    "sharp":       Profile(1, 20,  (2.0, 4.5), 0.04, vocab_cap=18000),
    "pro":         Profile(0, 8,   (1.2, 3.0), 0.012),
    "grandmaster": Profile(0, 30,  (0.9, 2.2), 0.0, strategic=True),
}


def choose_word(model, prev: str, used: set[str], difficulty: str,
                threshold: float, rng: random.Random | None = None,
                pressure: float = 0.0) -> str | None:
    """Return the bot's word, or None to freeze (time out).

    `pressure` in [0, 1] is how far the clock has tightened; mistakes get more
    likely under pressure, like they do for people.
    """
    rng = rng or random
    prof = PROFILES[difficulty]

    if rng.random() < prof.blunder * (1 + 3 * pressure) + (0.01 * pressure if prof.blunder else 0):
        kind = rng.choice(["forbidden", "forbidden", "repeat", "freeze"])
        if kind == "forbidden":
            bad = [w for w, _ in model.candidates(prev, exclude=used, k=40, allow_forbidden=True)
                   if w[0] in "rst"]
            if bad:
                return rng.choice(bad[:8])
        if kind == "repeat" and used:
            return rng.choice(sorted(used))
        return None

    cands = [(w, p) for w, p in model.candidates(prev, exclude=used, k=max(prof.rank_hi, 60) * 2)
             if p >= threshold and (prof.vocab_cap is None or model.index[w] < prof.vocab_cap)]
    if not cands:
        return None
    if prof.strategic:
        pool = cands[: prof.rank_hi]
        pressure_after = model.forbidden_pressure_many([w for w, _ in pool], used)
        scores = [0.55 * p + 0.45 * fp for (_, p), fp in zip(pool, pressure_after)]
        return pool[max(range(len(pool)), key=scores.__getitem__)][0]
    lo = min(prof.rank_lo, max(0, len(cands) - 1))
    pool = cands[lo: prof.rank_hi] or cands
    return rng.choice(pool)[0]


async def play_bot_turn(room, player, turn_no: int):
    prof = PROFILES.get(player.difficulty or "casual", PROFILES["casual"])
    limit = room.turn_length
    # Bots feel the clock too: think time shrinks with the turn length.
    scale = min(1.0, limit / max(1.0, room.settings.turn_seconds))
    delay = min(rng_uniform(*prof.think) * (0.5 + 0.5 * scale), max(0.6, limit - 1.0))
    try:
        await asyncio.sleep(delay)
    except asyncio.CancelledError:
        return
    # Think outside the lock so other players' messages aren't held up.
    if room.phase != "playing" or room.turn_no != turn_no:
        return
    thr = room.model.threshold(room.settings.strictness)
    # Disconnected humans are auto-played at casual level so the game keeps moving.
    diff = player.difficulty if player.is_bot else "casual"
    word = await asyncio.to_thread(choose_word, room.model, room.last_word, set(room.used), diff,
                                   thr, None, 1.0 - scale)
    if word is None:
        return  # freeze: the turn timer will expire
    async with room.lock:
        still_ours = (room.phase == "playing" and room.turn_no == turn_no
                      and room.current is player and (player.is_bot or not player.connected))
        if still_ours and time.time() <= (room.turn_deadline or 0):
            await room.submit(player.id, word)


def rng_uniform(a: float, b: float) -> float:
    return random.uniform(a, b)

## 2. Load artifacts
Upload `rst_artifacts.zip` when prompted (or set `ZIP` to a Drive path).

In [ ]:
import os, zipfile, json, random, time, itertools
from pathlib import Path
import numpy as np
ZIP = "rst_artifacts.zip"
if not Path(ZIP).exists():
    try:
        from google.colab import files
        ZIP = next(iter(files.upload()))
    except ImportError:
        raise SystemExit("Put rst_artifacts.zip next to this notebook.")
with zipfile.ZipFile(ZIP) as z:
    z.extractall("artifacts")
ART = next(p.parent for p in Path("artifacts").rglob("meta.json"))
from rst_nlp import AssociationModel
from rst_bots import PROFILES, choose_word
model = AssociationModel(ART)
print(model.meta["version"], "|", len(model.words), "words |", int(model.playable.sum()), "playable |",
      len(model.known) // 2, "known pairs")
print("thresholds:", model.meta["thresholds"])
t = time.perf_counter(); [model.candidates("coffee", k=60) for _ in range(50)]
print(f"bot candidate search: {(time.perf_counter()-t)/50*1000:.1f} ms")

## 3. Try the judge

In [ ]:
PAIRS = [("cat","dog"),("cat","purr"),("coffee","morning"),("ocean","wave"),("doctor","nurse"),
         ("bank","river"),("bank","money"),("hot","cold"),("bread","butter"),("moon","night"),
         ("cat","piano"),("apple","galaxy"),("chair","volcano"),("guitar","banana")]
thr = model.meta["thresholds"]
for a, b in PAIRS:
    p = model.relatedness(a, b)
    verdict = "strict" if p >= thr["strict"] else "normal" if p >= thr["normal"] else "lenient" if p >= thr["lenient"] else "reject"
    print(f"{a:>8} -> {b:<8} {p:.3f}  accepted at: {verdict}")

In [ ]:
def judge(a, b):
    a, b = a.lower().strip(), b.lower().strip()
    if not model.in_vocab(b): return f"'{b}' is not in the dictionary"
    if b[0] in "rst": return f"'{b}' starts with {b[0].upper()} - banned"
    return f"{a} -> {b}: {model.relatedness(a, b):.3f}"
for cue in ["ocean", "coffee", "winter", "music", "doctor"]:
    print(cue, "->", [w for w, _ in model.candidates(cue, k=12)])
print(judge("winter", "cold"))

## 4. Difficulty calibration by simulation

This mirrors the server rules: 3 lives, a miss costs a life and passes the same word on,
no repeats, and a clock that shrinks 4% per accepted word (15 s start, 4 s floor). A bot
"freezes" if it returns no word, and its think time is compared with the clock.

In [ ]:
BASE, MIN_TURN, SPEEDUP = 15.0, 4.0, 0.96
def simulate(levels, lives=3, strictness="normal", seed=0, max_turns=400):
    rng = random.Random(seed)
    thr = model.threshold(strictness)
    seedw = rng.choice(model.seeds)
    chain, used = [seedw], {model.normalize(seedw)}
    life = [lives] * len(levels); stats = [dict(turns=0, ok=0, miss=0, probs=[]) for _ in levels]
    turn = rng.randrange(len(levels)); n = 0
    while sum(l > 0 for l in life) > 1 and n < max_turns:
        n += 1
        length = max(MIN_TURN, BASE * SPEEDUP ** (len(chain) - 1))
        scale = min(1.0, length / BASE)
        prof = PROFILES[levels[turn]]
        think = rng.uniform(*prof.think) * (0.5 + 0.5 * scale)
        w = choose_word(model, chain[-1], used, levels[turn], thr, rng=rng, pressure=1 - scale)
        st = stats[turn]; st["turns"] += 1
        ok = (w is not None and think <= length and w[0] not in "rst" and model.in_vocab(w)
              and model.normalize(w) not in used and model.relatedness(chain[-1], w) >= thr)
        if ok:
            st["ok"] += 1; st["probs"].append(model.relatedness(chain[-1], w))
            chain.append(w); used.add(model.normalize(w))
        else:
            st["miss"] += 1; life[turn] -= 1
        for step in range(1, len(levels) + 1):
            j = (turn + step) % len(levels)
            if life[j] > 0: turn = j; break
    alive = [i for i, l in enumerate(life) if l > 0]
    return (alive[0] if len(alive) == 1 else None), stats, chain

LEVELS = list(PROFILES)
agg = {l: dict(turns=0, ok=0, probs=[]) for l in LEVELS}
for g in range(300):
    _, stats, _ = simulate(LEVELS, seed=g)
    for l, s in zip(LEVELS, stats):
        agg[l]["turns"] += s["turns"]; agg[l]["ok"] += s["ok"]; agg[l]["probs"] += s["probs"]
print(f"{'level':<12}{'success rate':>14}{'mean link prob':>16}")
for l in LEVELS:
    a = agg[l]
    print(f"{l:<12}{a['ok']/max(1,a['turns']):>14.1%}{np.mean(a['probs']):>16.3f}")

In [ ]:
# Head-to-head: row level vs column level, 200 one-on-one games each.
import pandas as pd
wins = pd.DataFrame(0.0, index=LEVELS, columns=LEVELS)
for a, b in itertools.permutations(LEVELS, 2):
    w = sum(simulate([a, b], seed=1000 + g)[0] == 0 for g in range(200))
    wins.loc[a, b] = w / 200
print("Win rate of ROW against COLUMN (1v1):")
display(wins.style.format("{:.0%}").background_gradient(cmap="RdYlGn", vmin=0, vmax=1))

In [ ]:
winner, stats, chain = simulate(["casual", "pro", "grandmaster"], seed=7)
print("sample game chain:", " -> ".join(chain[:40]))

A well-calibrated ladder has each level beating the one below it more often than not, and
Grandmaster beating Rookie almost always. If two adjacent levels come out near 50%, adjust
`PROFILES` in `server/bots.py` (rank window, blunder rate, think time) and re-run.